# Random Forest Model

This notebook trains the Random Forest model for the clinical trial outcome classifier. The first pass checks whether very high performance is being driven by features that would not be available near trial start. The saved model uses the cleaned feature set.

Input: `data/model_ready.parquet`  
Output: `artifacts/models/random_forest.joblib` and `artifacts/model_metrics.json`

## 1. Setup

The split matches the rest of the project: train on trials starting before 2019 and test on trials from 2019 onward.

In [1]:
import time

import pandas as pd
import polars as pl
from sklearn.ensemble import RandomForestClassifier

from modeling_utils import (
    LEAKAGE_PRONE_COLS,
    RANDOM_STATE,
    evaluate_model,
    load_model_ready,
    remove_leakage_prone_features,
    save_model,
    save_or_update_metrics,
    temporal_split,
)

## 2. Load Model-Ready Data

In [2]:
df, feature_cols = load_model_ready()
x_train, x_test, y_train, y_test = temporal_split(df, feature_cols)

print(f"Loaded: {df.shape[0]:,} rows x {len(feature_cols):,} features")
print(f"Train:  {x_train.shape[0]:,} rows")
print(f"Test:   {x_test.shape[0]:,} rows")

Loaded: 53,628 rows x 147 features
Train:  40,209 rows
Test:   13,419 rows


## 3. Leakage Check

The proposal frames the model as a prediction tool using information available at or near trial start. Features like `trial_duration_days`, `enrollment_actual`, and `log_enrollment` can encode information that is only known after a trial has progressed or finished. We train a quick full-feature model first so we can verify whether those fields dominate the result.

In [3]:
rf_full = RandomForestClassifier(
    n_estimators=250,
    max_depth=20,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    n_jobs=1,
    random_state=RANDOM_STATE,
)

start = time.perf_counter()
rf_full.fit(x_train, y_train)
full_training_seconds = time.perf_counter() - start

full_metrics = evaluate_model(
    model=rf_full,
    name="Random Forest Full Feature Audit",
    slug="random_forest_full_audit",
    x_train=x_train,
    x_test=x_test,
    y_test=y_test,
    training_seconds=full_training_seconds,
)

print(f"Full-feature Macro F1: {full_metrics['macro_f1']:.4f}")
for label in ["COMPLETED", "TERMINATED", "WITHDRAWN"]:
    scores = full_metrics["classification_report"][label]
    print(f"{label}: precision={scores['precision']:.4f}, recall={scores['recall']:.4f}, f1={scores['f1-score']:.4f}")

Full-feature Macro F1: 0.8500
COMPLETED: precision=0.8202, recall=0.7645, f1=0.7913
TERMINATED: precision=0.7289, recall=0.7907, f1=0.7586
WITHDRAWN: precision=1.0000, recall=1.0000, f1=1.0000


In [4]:
full_importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_full.feature_importances_,
}).sort_values("importance", ascending=False)

full_importances.head(20)

,feature,importance
26,log_enrollment,0.636538
32,num_sites,0.066344
48,trial_duration_days,0.022403
29,num_secondary_outcomes,0.017346
42,age_range_years,0.017276
41,max_age_years,0.014809
45,healthy_volunteers,0.013747
37,sponsor_class_OTHER,0.011793
40,min_age_years,0.011513
97,approval_year,0.010782


The full-feature model can score unrealistically well if it relies on leakage-prone variables (e.g. log enrollment). The final Random Forest below removes those fields and is the version saved for the dashboard.

This design choice also makes sense from a real deployment perspective. Hospitals or drug companies won't know the total log enrollment at the starting years of the clinical trial, so this will not be an appropriate feature for prediction.

## 4. Train Clean Random Forest

In [5]:
clean_feature_cols = remove_leakage_prone_features(feature_cols)
x_train_clean, x_test_clean, y_train_clean, y_test_clean = temporal_split(df, clean_feature_cols)

print(f"Removed leakage-prone columns: {sorted(LEAKAGE_PRONE_COLS)}")
print(f"Clean feature count: {len(clean_feature_cols):,}")

Removed leakage-prone columns: ['enrollment_actual', 'log_enrollment', 'trial_duration_days']
Clean feature count: 144


In [6]:
rf_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=20,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    n_jobs=1,
    random_state=RANDOM_STATE,
)

start = time.perf_counter()
rf_model.fit(x_train_clean, y_train_clean)
training_seconds = time.perf_counter() - start

print(f"Training complete in {training_seconds:.1f} seconds")

Training complete in 36.1 seconds


## 5. Evaluate and Save

In [7]:
metrics = evaluate_model(
    model=rf_model,
    name="Random Forest",
    slug="random_forest",
    x_train=x_train_clean,
    x_test=x_test_clean,
    y_test=y_test_clean,
    training_seconds=training_seconds,
)

save_model(rf_model, "random_forest")
save_or_update_metrics(metrics)

print(f"Macro F1:    {metrics['macro_f1']:.4f}")
print(f"Accuracy:    {metrics['accuracy']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
for label in ["COMPLETED", "TERMINATED", "WITHDRAWN"]:
    scores = metrics["classification_report"][label]
    print(f"{label}: precision={scores['precision']:.4f}, recall={scores['recall']:.4f}, f1={scores['f1-score']:.4f}")
print("Saved Random Forest artifacts.")

Macro F1:    0.5879
Accuracy:    0.6140
Weighted F1: 0.6117
COMPLETED: precision=0.6920, recall=0.6051, f1=0.6457
TERMINATED: precision=0.5597, recall=0.7121, f1=0.6267
WITHDRAWN: precision=0.5697, recall=0.4320, f1=0.4914
Saved Random Forest artifacts.


## 6. Clean Model Feature Importance

In [8]:
clean_importances = pd.DataFrame({
    "feature": clean_feature_cols,
    "importance": rf_model.feature_importances_,
}).sort_values("importance", ascending=False)

clean_importances.head(20)

,feature,importance
30,num_sites,0.206486
27,num_secondary_outcomes,0.057178
40,age_range_years,0.049942
39,max_age_years,0.045256
38,min_age_years,0.035348
94,approval_year,0.035263
29,num_drugs,0.031138
26,num_primary_outcomes,0.029953
43,healthy_volunteers,0.029423
32,sponsor_class_INDUSTRY,0.028770
